# Data Fusion 2025 — Задача 2 "4cast"
## Прогнозирование еженедельных переводов юридических лиц

**Задача:** предсказать суммарные переводы 51 963 клиентов банка за 12 недель (118–129),  
используя историю за 118 недель (0–117).

**Метрика:** средний по клиентам RMSLE:
$$\overline{RMSLE} = \frac{1}{N}\sum_{i=1}^N \sqrt{\frac{1}{T}\sum_{t=1}^T (\log(1+y_{it}) - \log(1+\hat{y}_{it}))^2}$$

**Public leaderboard:** недели 118–121 (4 из 12 прогнозных).

---


## Скачивание данных

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import lightgbm as lgb
import warnings
import json
from pathlib import Path
import gc, time

warnings.filterwarnings("ignore")
plt.rcParams["figure.figsize"] = (14, 5)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.4
sns.set_palette("tab10")

DATA_DIR = Path("data")
FIG_DIR  = Path("figures1")
FIG_DIR.mkdir(exist_ok=True)

# Splits (из calendar_extended.csv)
TRAIN_WEEKS    = list(range(0, 106))     # 0–105  → train
VAL_PUB_WEEKS  = list(range(106, 110))  # 106–109 → validation_public
VAL_PRIV_WEEKS = list(range(110, 118))  # 110–117 → validation_private
PUBLIC_WEEKS   = list(range(118, 122))  # 118–121 → public leaderboard
PRIVATE_WEEKS  = list(range(122, 130))  # 122–129 → private leaderboard
FORECAST_WEEKS = list(range(118, 130))  # все 12 недель прогноза

print("Config loaded.")


Config loaded.


In [2]:
print("Loading data...")
ts_train = pd.read_parquet(DATA_DIR / "target_series_extended.parquet")       # weeks 0–117
calendar = pd.read_csv(DATA_DIR / "calendar_extended.csv", parse_dates=["date"])
sample   = pd.read_csv(DATA_DIR / "sample_submit_extended.csv")
profiles = pd.read_parquet(DATA_DIR / "profiles_extended.parquet")

ts_train["week"] = ts_train["week"].astype(int)

TX_FILES = sorted(DATA_DIR.glob("transactions_[1-5].parquet"))
print(f"Found {len(TX_FILES)} transaction files: {[f.name for f in TX_FILES]}")

t0 = time.time()
trns_chunks = []
for f in TX_FILES:
    print(f"Reading {f.name} ...")
    chunk = pd.read_parquet(f)
    print(f"  -> {len(chunk):,} rows")
    trns_chunks.append(chunk)

trns_raw = pd.concat(trns_chunks, ignore_index=True)
del trns_chunks
gc.collect()

Loading data...
Found 5 transaction files: ['transactions_1.parquet', 'transactions_2.parquet', 'transactions_3.parquet', 'transactions_4.parquet', 'transactions_5.parquet']
Reading transactions_1.parquet ...
  -> 52,089,892 rows
Reading transactions_2.parquet ...
  -> 55,045,806 rows
Reading transactions_3.parquet ...
  -> 59,510,350 rows
Reading transactions_4.parquet ...
  -> 61,492,247 rows
Reading transactions_5.parquet ...
  -> 30,036,951 rows


0

## Подготовка данных 

In [21]:
trns = trns_raw.copy()

In [22]:
trns["date"] = pd.to_datetime(trns["date"], utc=True).dt.tz_localize(None).dt.normalize()

all_inn = ts_train["inn_id"].unique()

cal_117 = calendar.loc[calendar["week"] <= 117].copy()
cal_117["date"] = pd.to_datetime(cal_117["date"]).dt.tz_localize(None).dt.normalize()
all_dates = np.sort(cal_117["date"].drop_duplicates().values)

mi = pd.MultiIndex.from_product([all_inn, all_dates], names=["doc_payer_inn", "date"])
new_trns = mi.to_frame(index=False)

new_trns["doc_payee_inn"] = ""
new_trns["trns_count"] = 1.0
new_trns["trns_amount"] = 0
new_trns["doc_payer_bank_name_encoded"] = 51
new_trns["doc_payee_bank_name_encoded"] = 0
new_trns["doc_payer_bank_name_flag"] = 1
new_trns["doc_payee_bank_name_flag"] = 0
new_trns["trns_class_encoded"] = 0

trns = pd.concat([trns, new_trns], ignore_index=True)

trns = pd.merge(trns, calendar, on='date', how='left')

allowed = ts_train["inn_id"].unique()
trns = trns[trns["doc_payer_inn"].isin(allowed)]

In [23]:
# построение ряда с суммами переводов по неделям только со счетов втб 
# trns_vtb = trns[trns["doc_payer_bank_name_flag"] == 1].copy()
# trns_vtb = trns_vtb.groupby(["week", "doc_payer_inn"], as_index=False, sort=False).agg({
#     "trns_amount": "sum",
#     "part": "first",
# }).rename(columns={
#                  "doc_payer_inn": "inn_id",
#                  "trns_amount": "target",
#              })

# trns_vtb.drop(columns=["part"], inplace=True)

In [24]:
trns = trns.groupby(["week", "doc_payer_inn"], as_index=False, sort=False).agg({
    "trns_amount": "sum",
    "part": "first",
}).rename(columns={
                 "doc_payer_inn": "inn_id",
                 "trns_amount": "target",
             })

In [25]:
# trns = pd.merge(trns, trns_vtb, on=["week", "inn_id"], how="left")  


In [26]:
trns.head()

,week,inn_id,target,part
0,0,inn1000051,4.221593e+05,train
1,0,inn1000367,3.994532e+05,train
2,0,inn1000484,9.672847e+05,train
3,0,inn1000624,1.684209e+05,train
4,0,inn1000884,5.724560e+06,train


In [27]:
# del trns_vtb
del new_trns
del cal_117
gc.collect()

403

In [ ]:
expected_weeks = sorted(trns["week"].astype(int).unique().tolist())

expected_len = trns["inn_id"].nunique()

dup_mask = trns.duplicated(subset=["inn_id", "week"], keep=False)
if dup_mask.any():
    dup_count = int(dup_mask.sum())
    raise ValueError(f"Найдены дубли по (inn_id, week): {dup_count} строк")

min_week, max_week = expected_weeks[0], expected_weeks[-1]
full_range = list(range(min_week, max_week + 1))
if expected_weeks != full_range:
    missing_weeks = sorted(set(full_range) - set(expected_weeks))
    raise ValueError(f"Неполный диапазон недель. Отсутствуют: {missing_weeks}")

out_dir = Path("transactions_chunks")
out_dir.mkdir(parents=True, exist_ok=True)

trns_sorted = trns.sort_values(["week", "inn_id"]).reset_index(drop=True)
errors = []
written_files = 0

for week, chunk in trns_sorted.groupby("week", sort=True):
    chunk_len = len(chunk)
    if chunk_len != expected_len:
        errors.append(f"week={int(week)}: len={chunk_len}, expected={expected_len}")
        continue

    out_path = out_dir / f"week_{int(week):03d}.parquet"
    chunk.to_parquet(out_path, index=False)
    written_files += 1

if errors:
    preview = "\n".join(errors[:10])
    raise ValueError(f"Обнаружены недели с неверной длиной чанка:\n{preview}")

print(f"Done: записано {written_files} файлов в {out_dir.resolve()}")

Done: записано 118 файлов в /Users/aeseverenkova/Desktop/bank_forecasting/transactions_chunks


## Построение признаков

### Категориальные признаки

In [28]:
from sklearn.preprocessing import OrdinalEncoder
import joblib

In [29]:
pr = profiles.copy()
pr.isna().sum()

inn_id                                0
report_date                           0
ipul                                  0
id_region                       1141970
main_okved_group                1299647
diff_datopen_report_date_flg          0
dtype: int64

In [30]:
pr['id_region'].isna().sum() / pr['id_region'].count() * 100

np.float64(3.066721528945321)

In [31]:
pr['main_okved_group'].isna().sum() / pr['main_okved_group'].count() * 100

np.float64(3.5049991762363177)

In [32]:
# todo: изобразить на графике количество уникальных значений в profiles
# цель:показать, что уникальных значений не много и что сложный кодировщик не нужен 

In [33]:
# todo: распределение записей по inn_id
# цель: показать, что inn_id не изменяются во времени


Используем OrdinalEncoder, так как он часто используется с моделью XGBoost, подходит для категориальных колонок с умеренным разбросом значений и компактно кодирует признаки. (При увеличесии выборки начались проблемы с нехваткой памяти, поэтому нужно экономить)

Пустые значения считаются отдельной категорией. 

In [34]:
allowed = ts_train["inn_id"].unique()
cat_cols = ["ipul", "id_region", "main_okved_group", "diff_datopen_report_date_flg"]

prof = (
    profiles[profiles["inn_id"].isin(allowed)]
    .sort_values("report_date")
    .groupby("inn_id", as_index=False)
    .last()
)
prof[cat_cols] = prof[cat_cols].fillna("unknown").astype("string")

enc = OrdinalEncoder(
    handle_unknown="use_encoded_value",
    unknown_value=-1,
    dtype=np.int32,
)
prof[cat_cols] = enc.fit_transform(prof[cat_cols]).astype(np.int32)

joblib.dump(enc, DATA_DIR / "ordinal_encoder.joblib")

prof = prof.drop(columns=["report_date"], errors="ignore")
prof.head()

,inn_id,ipul,id_region,main_okved_group,diff_datopen_report_date_flg
0,inn1000051,0,34,59,4
1,inn1000246,1,22,15,6
2,inn1000281,1,29,18,8
3,inn1000367,1,30,23,4
4,inn1000409,0,34,59,8


### Числовые признаки

Хотим в признаках зафиксировать связь предыдущих событий на будующие. Для этого будем использовать лаги.

In [35]:
def _lags_row_at_cutoff(n: int, weeks: np.ndarray, y: np.ndarray, max_lag: int = 70):
    """Одна строка лагов для недели n (эквивалент shift)"""

    mask = (weeks <= n) & (weeks >= n - max_lag)
    idx = np.flatnonzero(mask)

    w = weeks[idx] # отфильтрованные недели в нужном диапозоне 
    yy = y[idx] # отфильтрованные таргеты в нужном диапозоне 
    pos = int(np.where(w == n)[0][0])

    row = {}
    for lag in range(1, max_lag + 1):
        j = pos - lag
        row[f"lag_{lag}"] = float(yy[j])
    return row

def get_features(n, df) -> pd.DataFrame:
    """
    n - номер недели
    df - датафрейм с транзакциями одного inn_id
    """

    max_lag = 70

    g = df.sort_values("week")
    weeks = g["week"].to_numpy()
    y = g["target"].to_numpy(dtype=float)
    row = _lags_row_at_cutoff(n, weeks, y, max_lag)
    
    return pd.DataFrame([row]) if row else pd.DataFrame()


In [36]:
def build_features_horizons_for_inn(
    n_target: int, k: int, inn_df: pd.DataFrame, max_lag: int = 70
) -> pd.DataFrame:
    """Все горизонты для одного ИНН """

    g = inn_df.sort_values("week")
    weeks = g["week"].to_numpy()
    y = g["target"].to_numpy(dtype=float)
    # full_amount = g["trns_amount"].to_numpy(dtype=float)

    rows = []
    for i in range(k):
        cutoff = n_target - i
        row = _lags_row_at_cutoff(cutoff, weeks, y, max_lag)
        row['inn_id'] = inn_df['inn_id'].iloc[0] # чтобы потом по inn доклеить profiles 
        rows.append(row) 
        
    return pd.DataFrame(rows) if rows else pd.DataFrame()

Представим, что мы хотим предсказать на одну неделю вперед. У нас есть данные 0-117 недель. Тогда мы будем считать 117 неделю таргетом, а строить признаки на 0-116 неделях. 

Если мы хотим предсказать 2 неделю, таргетом снова считаем 117 неделю, но признаки уже строим на 0-115 неделях. 

Если мы хотим предсказать 3 неделю, мы строрим признаки на 0-114 неделях. 

Тогда чтобы предсказать неделю k надо строить признаки на неделях 117-k. Так мы будем показывать как влияют предыдущие значения на будующую неделю номер k. 

Предсказание на 12 недель: предсказание на 1, предсказание на 2, .... предсказание на 12 неделю.

In [28]:
train = pd.DataFrame()
n = 117 # текущая неделя 
k = 12 # горизонт предсказания

train_list = []

df_week_n = trns.loc[trns["week"] == n, ["inn_id", "target"]]
inn_subset = df_week_n["inn_id"].unique()
inn_groups = {gid: g for gid, g in trns.groupby("inn_id", sort=False)}

train_list = []
for inn in inn_subset:
    inn_df = inn_groups.get(inn)

    t_series = df_week_n.loc[df_week_n["inn_id"] == inn, "target"]
    t_val = t_series.iloc[0]

    f_block = build_features_horizons_for_inn(n, k, inn_df)
    f_block["target"] = t_val
    train_list.append(f_block)
        
train = pd.concat(train_list, ignore_index=True) if train_list else pd.DataFrame() 

# добавление категориальных признаков 
train = train.merge(prof, on="inn_id", how="left")
for col in cat_cols:
    train[col] = train[col].fillna(-1).astype("category")
train = train.drop(columns=["inn_id"], errors="ignore")

# логарифмирование 
lag_target_cols = [c for c in train.columns if c.startswith("lag_") or c == "target" or c == 'trns_amount']
train[lag_target_cols] = train[lag_target_cols].applymap(lambda x: np.log(x + 1))

train.head()


,lag_1,lag_2,lag_3,lag_4,lag_5,lag_6,lag_7,lag_8,lag_9,lag_10,...,lag_66,lag_67,lag_68,lag_69,lag_70,target,ipul,id_region,main_okved_group,diff_datopen_report_date_flg
0,11.820790,11.317419,12.100341,12.405977,10.357664,0.000000,11.585395,14.735380,8.964775,10.961955,...,9.581792,11.399904,12.089898,11.090265,10.345816,10.999625,0,34,59,4
1,11.317419,12.100341,12.405977,10.357664,0.000000,11.585395,14.735380,8.964775,10.961955,11.974569,...,11.399904,12.089898,11.090265,10.345816,13.371397,10.999625,0,34,59,4
2,12.100341,12.405977,10.357664,0.000000,11.585395,14.735380,8.964775,10.961955,11.974569,15.821142,...,12.089898,11.090265,10.345816,13.371397,10.290305,10.999625,0,34,59,4
3,12.405977,10.357664,0.000000,11.585395,14.735380,8.964775,10.961955,11.974569,15.821142,11.487807,...,11.090265,10.345816,13.371397,10.290305,13.776555,10.999625,0,34,59,4
4,10.357664,0.000000,11.585395,14.735380,8.964775,10.961955,11.974569,15.821142,11.487807,14.610071,...,10.345816,13.371397,10.290305,13.776555,10.988557,10.999625,0,34,59,4


In [ ]:
del train
gc.collect()

Чтобы увеличить обучающую выборку, попробуем не фиксировать таргетную неделю на n = 117. Построим по схеме выше признаки для 10 таргенных недель (117...97). 

In [38]:
def sample_n(n, df) -> pd.DataFrame:
    """ Собрать обучающую выборку для недели n """
    k = 12 # горизонт предсказания

    df_week_n = df.loc[df["week"] == n, ["inn_id", "target"]]
    inn_subset = df_week_n["inn_id"].unique()
    inn_groups = {gid: g for gid, g in df.groupby("inn_id", sort=False)}

    train_list = []
    for inn in inn_subset:
        inn_df = inn_groups.get(inn)

        t_series = df_week_n.loc[df_week_n["inn_id"] == inn, "target"]
        t_val = t_series.iloc[0]

        f_block = build_features_horizons_for_inn(n, k, inn_df)
        f_block["target"] = t_val
        train_list.append(f_block)

    return pd.concat(train_list, ignore_index=True) if train_list else pd.DataFrame()

In [39]:
import time

start_time = time.perf_counter()

N = 10
train_parts = []

for i in range(N):
    n = 117 - i
    train_parts.append(sample_n(n, trns))

train = pd.concat(train_parts, ignore_index=True) if train_parts else pd.DataFrame()

# добавление категориальных признаков 
train = train.merge(prof, on="inn_id", how="left")
for col in cat_cols:
    train[col] = train[col].fillna(-1).astype("category")
train = train.drop(columns=["inn_id"], errors="ignore")

# логарифмирование 
lag_target_cols = [c for c in train.columns if c.startswith("lag_") or c == "target" or c == 'trns_amount']
train[lag_target_cols] = train[lag_target_cols].applymap(lambda x: np.log(x + 1))

end_time = time.perf_counter()
execution_time = (end_time - start_time) / 60

print(f"Время выполнения: {execution_time:.2f} минут")

train.head()

Время выполнения: 18.29 минут


,lag_1,lag_2,lag_3,lag_4,lag_5,lag_6,lag_7,lag_8,lag_9,lag_10,...,lag_66,lag_67,lag_68,lag_69,lag_70,target,ipul,id_region,main_okved_group,diff_datopen_report_date_flg
0,11.820790,12.645852,13.426701,12.470371,10.357664,10.593899,15.600336,14.735380,9.982253,10.961955,...,9.581792,11.399904,12.089898,11.090265,13.048860,11.195589,0,34,59,4
1,12.645852,13.426701,12.470371,10.357664,10.593899,15.600336,14.735380,9.982253,10.961955,12.979310,...,11.399904,12.089898,11.090265,13.048860,13.371397,11.195589,0,34,59,4
2,13.426701,12.470371,10.357664,10.593899,15.600336,14.735380,9.982253,10.961955,12.979310,15.886740,...,12.089898,11.090265,13.048860,13.371397,10.290305,11.195589,0,34,59,4
3,12.470371,10.357664,10.593899,15.600336,14.735380,9.982253,10.961955,12.979310,15.886740,15.084946,...,11.090265,13.048860,13.371397,10.290305,13.776555,11.195589,0,34,59,4
4,10.357664,10.593899,15.600336,14.735380,9.982253,10.961955,12.979310,15.886740,15.084946,14.614032,...,13.048860,13.371397,10.290305,13.776555,10.988557,11.195589,0,34,59,4


In [40]:
# с признаками по общему ряду переводов
# train.to_parquet('train_extended.parquet')

In [41]:
train.to_parquet('train.parquet')

## Обучение

In [42]:
!pip3 install xgboost

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip available: 22.3.1 -> 26.1.1
[notice] To update, run: pip3 install --upgrade pip


In [43]:
import xgboost as xgb
from sklearn.metrics import mean_squared_log_error

In [44]:
train = train.dropna()

y = train["target"]
X = train.drop(columns=["target"])

split = int(len(train) * 0.8)
X_train, X_test = X.iloc[:split], X.iloc[split:]
y_train, y_test = y.iloc[:split], y.iloc[split:]

model = xgb.XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="reg:squarederror",
    random_state=42,
    enable_categorical=True,
)

model.fit(X_train, y_train)
pred_log = model.predict(X_test)
pred = np.expm1(pred_log)


In [45]:
y_test = np.expm1(y_test)

In [46]:
rmsle = np.sqrt(mean_squared_log_error(y_test, pred))
print(f"RMSLE: {rmsle}")

RMSLE: 1.8286309718311475


In [47]:
pred

array([8265192.  , 8245974.  , 8278626.5 , ...,   69571.03,   37354.36,
         35078.73], shape=(1247112,), dtype=float32)